# 5. Energy and heat transport

**Learning goals.** In this tutorial you will:

- distinguish the particle, energy, and heat currents that QmeQ reports;
- verify the stationary conservation laws that relate them;
- drive transport with a temperature difference instead of a voltage;
- compute a thermovoltage and compare it with an exact first-order result; and
- convert the natural units used throughout into laboratory units.

Tutorials 1&ndash;4 followed charge. Thermoelectric measurements, heat engines, and refrigerators need the energy that the charge carries with it, which is a separate current with its own conservation law.

## Three currents

For every lead channel $\alpha$, QmeQ reports

- `system.current[a]` — the **particle current** $I_\alpha$, positive when particles flow from lead $\alpha$ into the dot;
- `system.energy_current[a]` — the **energy current** $J_\alpha$, the energy those particles carry, with the same sign convention;
- `system.heat_current[a]` — the **heat current**, the part of the energy flow that is not chemical work,

$$J^Q_\alpha=J_\alpha-\mu_\alpha I_\alpha.$$

In a stationary state the dot stores neither charge nor energy, so

$$\sum_\alpha I_\alpha=0,\qquad\sum_\alpha J_\alpha=0.$$

Together these give the first law for the device: the electrical power delivered to the leads is

$$P=-\sum_\alpha\mu_\alpha I_\alpha=\sum_\alpha J^Q_\alpha .$$

With $\mu_L=V/2$, $\mu_R=-V/2$ and $I_L=-I_R=I$ this is $P=-VI$, which is negative — dissipation — whenever the current flows in the direction the bias pushes it. A positive $P$ means the device is generating electrical power, and something else must be supplying the energy.

**Prediction before calculating.** With both a voltage and a temperature difference applied to the Anderson model, the reported `current`, `energy_current`, and `heat_current` must satisfy all three relations above to numerical precision.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qmeq
from scipy.optimize import brentq

print(qmeq.get_backend_status())

In [ ]:
U = 4.0
gamma = 0.1
tunnel_amplitude = np.sqrt(gamma / (2 * np.pi))
bandwidth = 40.0
base_temperature = 0.2

def make_anderson(kerntype, gate, bias, delta_temperature=0.0):
    """Spinful Anderson dot; the left channels (0, 2) are the hot ones."""
    hot = base_temperature + delta_temperature
    return qmeq.Builder(
        nsingle=2,
        hsingle={(0, 0): gate, (1, 1): gate},
        coulomb={(0, 1, 1, 0): U},
        nleads=4,
        tleads={(0, 0): tunnel_amplitude, (1, 0): tunnel_amplitude,
                (2, 1): tunnel_amplitude, (3, 1): tunnel_amplitude},
        mulst={0: bias / 2, 1: -bias / 2, 2: bias / 2, 3: -bias / 2},
        tlst={0: hot, 1: base_temperature, 2: hot, 3: base_temperature},
        dband=bandwidth,
        kerntype=kerntype,
    )

system = make_anderson("1vN", gate=-U / 2, bias=1.0, delta_temperature=0.1)
system.solve()

chemical_potentials = np.array([0.5, -0.5, 0.5, -0.5])
print("particle current:", system.current)
print("energy current:  ", system.energy_current)
print("heat current:    ", system.heat_current)

# The three relations of this section, as numerical checks.
assert np.allclose(system.heat_current,
                   system.energy_current - chemical_potentials * system.current)
assert np.isclose(np.sum(system.current), 0.0, atol=1e-12)
assert np.isclose(np.sum(system.energy_current), 0.0, atol=1e-12)

power = -np.dot(chemical_potentials, system.current)
print(f"\npower delivered to the leads: {power:.6e}")
print(f"sum of heat currents:        {np.sum(system.heat_current):.6e}")
assert np.isclose(power, np.sum(system.heat_current), atol=1e-12)

Note what the numbers say: the hot and cold sides do *not* exchange equal amounts of heat. Their difference is the electrical power, and with this bias and temperature difference the device is dissipating.

## Tight coupling: a first-order dot is a perfect energy filter

A first-order (sequential) approach lets an electron enter or leave only by a real transition between two dot eigenstates. The energy transferred to the lead is then exactly the energy difference of those states — nothing else is available. For a single level of energy $\varepsilon$ this means

$$J_\alpha=\varepsilon I_\alpha,\qquad J^Q_\alpha=(\varepsilon-\mu_\alpha)I_\alpha,$$

a relation known as *tight coupling*: heat flow and particle flow are locked together. It has an immediate and testable consequence — when the particle current vanishes, every heat current vanishes with it.

In [ ]:
level_gamma = 0.1
level_amplitude = np.sqrt(level_gamma / (2 * np.pi))
level_energy = 1.0
hot_temperature, cold_temperature = 1.0, 0.5

def make_single_level(kerntype, bias, energy=level_energy):
    return qmeq.Builder(
        nsingle=1,
        hsingle={(0, 0): energy},
        coulomb={},
        nleads=2,
        tleads={(0, 0): level_amplitude, (1, 0): level_amplitude},
        mulst={0: bias / 2, 1: -bias / 2},
        tlst={0: hot_temperature, 1: cold_temperature},
        dband=bandwidth,
        kerntype=kerntype,
    )

level = make_single_level("Pauli", bias=0.3)
level.solve()

print("particle current:", level.current)
print("energy current:  ", level.energy_current)
print("epsilon * I:     ", level_energy * level.current)
assert np.allclose(level.energy_current, level_energy * level.current, atol=1e-14)
print("\ntight coupling J = epsilon * I confirmed for a first-order kernel")

## Thermoelectric response and the thermovoltage

Now remove the voltage and keep only the temperature difference. Electrons above $\mu$ prefer to leave the hot side, electrons below it prefer to enter — so a level above the chemical potential and a level below it drive current in opposite directions. The voltage that has to be applied to stop the current is the **thermovoltage** $V_\mathrm{th}$, defined by $I(V_\mathrm{th})=0$; it is what an open-circuit thermoelectric measurement reports.

Tight coupling makes $V_\mathrm{th}$ exactly calculable for the single level. The current vanishes when the two reservoir occupations at the level coincide, $f_L(\varepsilon)=f_R(\varepsilon)$, i.e. when

$$\frac{\varepsilon-\mu_L}{T_L}=\frac{\varepsilon-\mu_R}{T_R}.$$

With $\mu_{L,R}=\pm V/2$, $T_L=T+\Delta T$, and $T_R=T$ this gives

$$V_\mathrm{th}=-\frac{2\varepsilon\,\Delta T}{2T+\Delta T}.$$

The linear slope in $\varepsilon$ is the hallmark of a tightly coupled energy filter, and Tutorial 6 shows how second-order processes break it.

In [ ]:
delta_temperature = hot_temperature - cold_temperature

def analytical_thermovoltage(energy):
    # T in the formula is the cold-side temperature.
    return (-2 * energy * delta_temperature
            / (2 * cold_temperature + delta_temperature))

level_energies = np.linspace(-3.0, 3.0, 25)
numerical = np.empty_like(level_energies)

for index, energy in enumerate(level_energies):
    thermoelectric = make_single_level("Pauli", bias=0.0, energy=energy)

    def current_at(bias):
        thermoelectric.change(mulst={0: bias / 2, 1: -bias / 2})
        thermoelectric.solve()
        return thermoelectric.current[0]

    numerical[index] = brentq(current_at, -20.0, 20.0, xtol=1e-12)

assert np.allclose(numerical, analytical_thermovoltage(level_energies), atol=1e-9)

fig, axis = plt.subplots(figsize=(5.5, 3.8))
axis.plot(level_energies, numerical, "ko", ms=4, label="QmeQ (Pauli)")
axis.plot(level_energies, analytical_thermovoltage(level_energies), "C0-",
          label="tight-coupling formula")
axis.set(xlabel="$\\varepsilon$", ylabel="$V_\\mathrm{th}$",
         title="Thermovoltage of a single level")
axis.legend()
fig.tight_layout()

print(f"largest deviation from the formula: "
      f"{np.max(np.abs(numerical - analytical_thermovoltage(level_energies))):.2e}")

At $V=V_\mathrm{th}$ the particle current is zero, and tight coupling forces the heat current to be zero as well: the device transports nothing at all. That is why a first-order calculation predicts a *stalled* engine to be reversible, and why it can be badly wrong about thermoelectric efficiency. The heat current at the stall point is the quantity to watch.

In [ ]:
stalled = make_single_level("Pauli", bias=analytical_thermovoltage(level_energy))
stalled.solve()

print("at V = V_th:")
print("  particle current:", stalled.current)
print("  heat current:    ", stalled.heat_current)
assert np.allclose(stalled.current, 0.0, atol=1e-12)
assert np.allclose(stalled.heat_current, 0.0, atol=1e-12)
print("\nfirst order: zero particle current implies zero heat current")

## Heat flow in the interacting dot

The interacting Anderson model has more than one transition energy, so the addition energies $\varepsilon$ and $\varepsilon+U$ can sit on opposite sides of the chemical potential. Sweeping the gate at fixed $\Delta T$ therefore reverses the thermoelectric current where each of them crosses $\mu=0$, and once more in between: at the particle-hole symmetric point $\varepsilon=-U/2$ the electron-like and hole-like contributions cancel exactly by symmetry. Expect three sign changes.

In [ ]:
gate_values = np.linspace(-1.6 * U, 0.6 * U, 111)
thermal_current = np.empty_like(gate_values)
hot_heat_current = np.empty_like(gate_values)

thermal = make_anderson("1vN", gate=gate_values[0], bias=0.0,
                        delta_temperature=0.3)
for index, gate in enumerate(gate_values):
    thermal.change(hsingle={(0, 0): gate, (1, 1): gate})
    thermal.solve()
    thermal_current[index] = thermal.current[0] + thermal.current[2]
    hot_heat_current[index] = thermal.heat_current[0] + thermal.heat_current[2]
    assert np.isclose(np.sum(thermal.energy_current), 0.0, atol=1e-12)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
axes[0].plot(gate_values / U, thermal_current / gamma, "k")
axes[0].axhline(0.0, color="0.7", lw=0.8)
axes[0].set(xlabel="$\\varepsilon/U$", ylabel="$I_\\mathrm{hot}/\\Gamma$",
            title="Thermoelectric particle current")
axes[1].plot(gate_values / U, hot_heat_current / gamma, "C1")
axes[1].axhline(0.0, color="0.7", lw=0.8)
axes[1].set(xlabel="$\\varepsilon/U$", ylabel="$J^Q_\\mathrm{hot}/\\Gamma$",
            title="Heat drawn from the hot side")
fig.tight_layout()

sign_changes = np.sum(np.diff(np.sign(thermal_current)) != 0)
print(f"sign changes of the thermoelectric current: {sign_changes}")
assert sign_changes == 3
assert abs(np.interp(-U / 2, gate_values, thermal_current)) < 1e-14
print(f"heat always leaves the hot reservoir: "
      f"{bool(np.all(hot_heat_current >= -1e-14))}")

The three zeros appear exactly where predicted: near $\varepsilon=0$ and $\varepsilon=-U$, where the carriers switch from electron-like to hole-like, and at $\varepsilon=-U/2$, where symmetry cancels the two contributions. The heat current, in contrast, never changes sign: heat always flows out of the hot reservoir, as the second law requires. A calculation that produced heat flowing *into* the hot side at zero bias would be reporting an error, not a discovery.

## Units

Everything above is in natural units $\hbar=k_\mathrm{B}=|e|=1$, with all energies measured against one chosen scale. Two conversions matter when comparing with experiment.

**Tunnelling amplitudes.** The amplitudes in `tleads` are density-of-states-weighted, $\mathsf{t}_{\alpha i}=\sqrt{\nu_F}\,t_{\alpha i}$, so the rate is simply $\Gamma_{\alpha i}=2\pi|\mathsf{t}_{\alpha i}|^2$ and $\nu_F$ never has to be specified. Channel-dependent densities of states are absorbed the same way.

**Restoring $e$ and $\hbar$.** QmeQ returns particle and energy currents. For carriers of charge $e$:

| quantity | natural units | laboratory units |
| --- | --- | --- |
| voltage, gate voltage | $V$, $V_g$ | $eV$, $eV_g$ |
| particle current | $I\;[\Gamma]$ | $I\;[e\Gamma/\hbar]$ |
| differential conductance | $\mathrm{d}I/\mathrm{d}V\;[1]$ | $\mathrm{d}I/\mathrm{d}V\;[e^2/\hbar]$ |
| energy and heat current | $J\;[\Gamma]$ | $J\;[\hbar\Gamma^2]$ |

To express conductance in units of $G_0=e^2/h$, multiply the $\mathrm{d}I/\mathrm{d}V$ values by $2\pi$. Note that the sign of the *electrical* current follows the carrier charge, so for electrons ($e<0$) it is opposite to the particle current reported here.

In [ ]:
conductance_quantum_factor = 2 * np.pi  # dI/dV in units of e^2/h

linear_response = make_anderson("1vN", gate=0.0, bias=0.0)
dV = 1e-4
linear_response.change(mulst={0: dV / 2, 1: -dV / 2, 2: dV / 2, 3: -dV / 2})
linear_response.solve()
conductance = (linear_response.current[0] + linear_response.current[2]) / dV

print(f"G = {conductance:.6f} in natural units")
print(f"  = {conductance * conductance_quantum_factor:.6f} e^2/h")

## Validity

The conservation laws checked here are properties of the stationary solution and hold for every approach QmeQ implements — they are useful as bug detectors, not as evidence that the approximation is good. Tight coupling is different: it is an artifact of first-order theory. Any prediction that leans on it — thermovoltage lineshapes, efficiency at low power, the reversibility of a stalled engine — must be re-examined with a second-order approach, which is what Tutorial 6 does.

## Exercises

1. Reduce `delta_temperature` in the Anderson sweep to `0.05`. The thermoelectric current shrinks; does it stay linear in $\Delta T$?
2. Increase `delta_temperature` to `1.0` (larger than $U/4$). Do the three sign changes survive?
3. Break the left/right coupling symmetry in `make_single_level` and recompute $V_\mathrm{th}$. The formula still holds: explain why the stall condition does not involve the coupling strengths, even though the current does.
4. Repeat the tight-coupling check with `kerntype="1vN"` and with a *two-level* dot. Tight coupling holds per transition, not per lead: which relation survives?